# Event-to-Visa Interactive Analysis

This notebook is a **user-facing analysis layer** over precomputed outputs generated by the script.

Run first (outside notebook):
`python src/analysis/event_visa_analysis.py --max-lag 6 --top-labels 8 --min-overlap 12 --min-event-months 12`

Notebook goals:
- Explore top event and sentiment lead/lag signals
- Filter countries and significance interactively
- View heatmaps and country overlay plots without recomputing the full pipeline

In [ ]:
from pathlib import Path
import polars as pl
import matplotlib.pyplot as plt
import seaborn as sns

try:
    widgets = __import__("ipywidgets")
except Exception:
    widgets = None

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (16, 7)

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

OUTPUT_DIR = PROJECT_ROOT / "data/processed/test_outputs"
PLOTS_DIR = PROJECT_ROOT / "data/plots/events_vs_visas"

def show_table(title: str, df: pl.DataFrame, rows: int = 20):
    print(f"\n=== {title} ===")
    if df.is_empty():
        print("(empty)")
        return
    with pl.Config(tbl_rows=rows, tbl_cols=18, tbl_width_chars=180, fmt_str_lengths=50):
        print(df.head(rows))

def read_parquet_safe(path: Path) -> pl.DataFrame:
    return pl.read_parquet(path) if path.exists() else pl.DataFrame()

def read_csv_safe(path: Path) -> pl.DataFrame:
    return pl.read_csv(path) if path.exists() else pl.DataFrame()

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")
print(f"PLOTS_DIR: {PLOTS_DIR}")
if widgets is None:
    print("ipywidgets not available: interactive controls will be skipped.")

In [ ]:
overlay_index = read_csv_safe(OUTPUT_DIR / "event_visa_overlay_index.csv")
lead_lag_results = read_parquet_safe(OUTPUT_DIR / "event_visa_lead_lag_results.parquet")
best_lags = read_parquet_safe(OUTPUT_DIR / "event_visa_best_lags.parquet")
monthly_sentiment = read_parquet_safe(OUTPUT_DIR / "event_monthly_sentiment.parquet")
sentiment_lag_results = read_parquet_safe(OUTPUT_DIR / "event_sentiment_lead_lag_results.parquet")
best_sentiment_lags = read_parquet_safe(OUTPUT_DIR / "event_sentiment_best_lags.parquet")

expected_outputs = [
    OUTPUT_DIR / "event_visa_overlay_index.csv",
    OUTPUT_DIR / "event_visa_lead_lag_results.parquet",
    OUTPUT_DIR / "event_visa_best_lags.parquet",
    OUTPUT_DIR / "event_monthly_sentiment.parquet",
    OUTPUT_DIR / "event_sentiment_lead_lag_results.parquet",
    OUTPUT_DIR / "event_sentiment_best_lags.parquet",
]
missing_outputs = [str(p) for p in expected_outputs if not p.exists()]

countries_event = set(best_lags["country"].to_list()) if best_lags.height else set()
countries_sent = set(best_sentiment_lags["country"].to_list()) if best_sentiment_lags.height else set()
countries = sorted(countries_event.union(countries_sent))

print(f"Loaded countries: {len(countries):,}")
print(f"Lead/lag rows: {lead_lag_results.height:,}")
print(f"Best event lags rows: {best_lags.height:,}")
print(f"Sentiment lag rows: {sentiment_lag_results.height:,}")
print(f"Best sentiment lags rows: {best_sentiment_lags.height:,}")
if missing_outputs:
    print("\nMissing outputs (run script mode first):")
    for p in missing_outputs:
        print(f"- {p}")

show_table("Countries available for analysis", pl.DataFrame({"country": countries}), rows=30)

In [ ]:
top_event_signals = (
    best_lags.sort(["q_value", "abs_corr"], descending=[False, True]).head(25)
    if best_lags.height
    else pl.DataFrame()
)
top_sentiment_signals = (
    best_sentiment_lags.sort(["q_value", "abs_corr"], descending=[False, True]).head(25)
    if best_sentiment_lags.height
    else pl.DataFrame()
)

show_table("Top event signals (best lags)", top_event_signals, rows=25)
show_table("Top sentiment signals (best lags)", top_sentiment_signals, rows=25)

## Overlay Plots for All Countries

This generates one plot per matched country with:
- stacked bars: monthly event counts by event label (top labels + Other)
- line: monthly visa issuances

In [ ]:
overlay_available = overlay_index.filter(pl.col("plot_path").is_not_null()) if overlay_index.height else pl.DataFrame()

print(f"Overlay rows: {overlay_index.height:,}")
print(f"Overlay plots available: {overlay_available.height:,}")
show_table("Overlay index sample", overlay_index, rows=20)

## Lead/Lag Analysis (All Labels)

For each country and each event label, evaluate lagged relationships from 0 to 6 months between event volume and visa issuance levels, then score event-surge to visa-surge alignment.

In [ ]:
if lead_lag_results.is_empty() or best_lags.is_empty():
    print("Lead/lag outputs are missing. Run script mode first.")
else:
    print(f"All lag rows: {lead_lag_results.height:,}")
    print(f"Best lag rows: {best_lags.height:,}")
    print(f"Significant event signals (q<=0.10): {best_lags.filter(pl.col('q_value') <= 0.10).height:,}")
    show_table("Best lag results preview", best_lags.sort(["q_value", "abs_corr"], descending=[False, True]), rows=30)

In [ ]:
all_label_candidates = (
    best_lags.filter((pl.col("q_value") <= 0.10) & (pl.col("surge_lift") >= 1.0))
    .sort(["q_value", "surge_lift", "abs_corr"], descending=[False, True, True])
)

show_table("All-label candidate signals", all_label_candidates, rows=30)

## Event Signal Deep Dive

Explore precomputed best event-label lags by significance and strength.

In [ ]:
top_best = (
    best_lags.filter(pl.col("q_value") <= 0.10)
    .sort(["q_value", "surge_lift", "abs_corr"], descending=[False, True, True])
    if best_lags.height
    else pl.DataFrame()
)

show_table("Top event lag signals (q<=0.10)", top_best, rows=30)

In [ ]:
focus_country = top_best["country"][0] if top_best.height else (countries[0] if countries else None)

if focus_country is None:
    print("No country available for visualization")
else:
    source_df = top_best if top_best.height else best_lags
    country_rows = source_df.filter(pl.col("country") == focus_country)
    sent = country_rows.select(["cluster_label", "lag_months", "pearson_corr"])

    pivot = sent.pivot(values="pearson_corr", index="cluster_label", on="lag_months", aggregate_function="max")
    lag_cols = [c for c in pivot.columns if c != "cluster_label"]

    if not lag_cols:
        print("No lag columns available for heatmap")
    else:
        pivot = pivot.sort(lag_cols)
        matrix = pivot.select(lag_cols).to_numpy()
        plt.figure(figsize=(14, max(6, 0.5 * max(1, pivot.height))))
        sns.heatmap(
            matrix,
            cmap="coolwarm",
            center=0,
            annot=False,
            yticklabels=pivot["cluster_label"].to_list(),
            xticklabels=lag_cols,
        )
        plt.title(f"{focus_country}: Event Label vs Lag Correlation with Visa Issuances")
        plt.xlabel("Lag (months)")
        plt.ylabel("Event Label")
        plt.tight_layout()
        plt.show()

## Sentiment Signal Deep Dive

Explore precomputed sentiment-lead/lag outputs and sentiment-shock alignment metrics.

In [ ]:
if monthly_sentiment.is_empty():
    print("Monthly sentiment output is missing. Run script mode first.")
else:
    print(f"Monthly sentiment rows: {monthly_sentiment.height:,}")
    show_table("Monthly sentiment preview", monthly_sentiment.sort(["country", "month"]).head(200), rows=20)

In [ ]:
if sentiment_lag_results.is_empty() or best_sentiment_lags.is_empty():
    print("Sentiment lag outputs are missing. Run script mode first.")
else:
    print(f"Sentiment lag rows: {sentiment_lag_results.height:,}")
    print(f"Best sentiment lag rows: {best_sentiment_lags.height:,}")
    print(f"Significant sentiment signals (q<=0.10): {best_sentiment_lags.filter(pl.col('q_value') <= 0.10).height:,}")
    show_table(
        "Best sentiment lag signals",
        best_sentiment_lags.sort(["q_value", "abs_corr"], descending=[False, True]),
        rows=30,
    )

In [ ]:
focus_sent_country = best_sentiment_lags["country"][0] if best_sentiment_lags.height else None

if focus_sent_country is None:
    print("No sentiment country available for visualization")
else:
    sent_rows = best_sentiment_lags.filter(pl.col("country") == focus_sent_country)
    sent = sent_rows.select(["cluster_label", "lag_months", "pearson_corr"])

    sent_pivot = sent.pivot(values="pearson_corr", index="cluster_label", on="lag_months", aggregate_function="max")
    lag_cols = [c for c in sent_pivot.columns if c != "cluster_label"]
    if lag_cols:
        sent_pivot = sent_pivot.sort(lag_cols)
        matrix = sent_pivot.select(lag_cols).to_numpy()
        plt.figure(figsize=(14, max(6, 0.5 * max(1, sent_pivot.height))))
        sns.heatmap(
            matrix,
            cmap="RdBu_r",
            center=0,
            annot=False,
            yticklabels=sent_pivot["cluster_label"].to_list(),
            xticklabels=lag_cols,
        )
        plt.title(f"{focus_sent_country}: Sentiment vs Visa Lag Correlation")
        plt.xlabel("Lag (months)")
        plt.ylabel("Event Label")
        plt.tight_layout()
        plt.show()
    else:
        print("No lag columns available for sentiment heatmap")

In [ ]:
overlay_generated = (
    int(overlay_index.select(pl.col("plot_path").is_not_null().sum()).item()) if overlay_index.height else 0
)

summary_metrics = pl.DataFrame(
    {
        "metric": [
            "countries_available",
            "overlay_rows",
            "overlay_plots_available",
            "lead_lag_rows",
            "best_event_lags_rows",
            "significant_event_signals_q<=0.10",
            "monthly_sentiment_rows",
            "sentiment_lag_rows",
            "best_sentiment_lags_rows",
            "significant_sentiment_signals_q<=0.10",
        ],
        "value": [
            len(countries),
            overlay_index.height,
            overlay_generated,
            lead_lag_results.height,
            best_lags.height,
            best_lags.filter(pl.col("q_value") <= 0.10).height if best_lags.height else 0,
            monthly_sentiment.height,
            sentiment_lag_results.height,
            best_sentiment_lags.height,
            best_sentiment_lags.filter(pl.col("q_value") <= 0.10).height if best_sentiment_lags.height else 0,
        ],
    }
)

show_table("Analysis coverage summary", summary_metrics, rows=30)

## Interactive Explorer

Use controls to explore best lag signals by country and mode (`event` or `sentiment`) without rerunning all analysis.

In [ ]:
event_cols = [
    "country", "cluster_label", "lag_months", "pearson_corr",
    "q_value", "surge_lift", "surge_precision", "surge_recall", "n_months",
]
sent_cols = [
    "country", "cluster_label", "lag_months", "pearson_corr",
    "q_value", "neg_shock_lift", "neg_shock_precision", "pos_shock_lift", "n_months",
]

event_best_view = best_lags.select(event_cols) if best_lags.height else pl.DataFrame(schema=[(c, pl.Float64 if c not in {"country", "cluster_label"} else pl.String) for c in event_cols])
sent_best_view = best_sentiment_lags.select(sent_cols) if best_sentiment_lags.height else pl.DataFrame(schema=[(c, pl.Float64 if c not in {"country", "cluster_label"} else pl.String) for c in sent_cols])

overlay_lookup = {}
if overlay_index.height:
    overlay_lookup = dict(zip(overlay_index["country"].to_list(), overlay_index["plot_path"].to_list()))

def show_country_heatmap(df: pl.DataFrame, country: str, title_prefix: str):
    rows = df.filter(pl.col("country") == country)
    if rows.is_empty():
        print(f"No rows available for {country}")
        return
    pivot = rows.select(["cluster_label", "lag_months", "pearson_corr"]).pivot(
        values="pearson_corr", index="cluster_label", on="lag_months", aggregate_function="max"
    )
    lag_cols = [c for c in pivot.columns if c != "cluster_label"]
    if not lag_cols:
        print("No lag columns available for heatmap")
        return
    pivot = pivot.sort(lag_cols)
    matrix = pivot.select(lag_cols).to_numpy()
    plt.figure(figsize=(13, max(5, 0.4 * max(1, pivot.height))))
    sns.heatmap(
        matrix,
        cmap="coolwarm",
        center=0,
        annot=False,
        yticklabels=pivot["cluster_label"].to_list(),
        xticklabels=lag_cols,
    )
    plt.title(f"{title_prefix} | {country}")
    plt.xlabel("Lag (months)")
    plt.ylabel("Event Label")
    plt.tight_layout()
    plt.show()

In [ ]:
if widgets is None:
    print("Install ipywidgets to use interactive explorer: pip install ipywidgets")
else:
    from IPython.display import display
    from IPython.display import Image as IPyImage

    mode = widgets.ToggleButtons(
        options=[("Event", "event"), ("Sentiment", "sentiment")],
        value="event",
        description="Mode:",
    )
    countries = sorted(set(best_lags["country"].to_list()) | set(best_sentiment_lags["country"].to_list()))
    country = widgets.Dropdown(options=countries, value=countries[0] if countries else None, description="Country:")
    q_max = widgets.FloatSlider(value=0.10, min=0.001, max=0.50, step=0.001, description="q ≤", readout_format=".3f")
    top_n = widgets.IntSlider(value=15, min=5, max=50, step=1, description="Top N")

    output = widgets.Output()

    def _render(*_):
        with output:
            output.clear_output(wait=True)
            if country.value is None:
                print("No countries available")
                return

            if mode.value == "event":
                base = event_best_view.filter(pl.col("country") == country.value)
                filtered = (
                    base.filter(pl.col("q_value") <= q_max.value)
                    .sort(["q_value", "surge_lift", "pearson_corr"], descending=[False, True, True])
                    .head(top_n.value)
                )
                show_table(f"Event signals | {country.value}", filtered, rows=top_n.value)
                show_country_heatmap(best_lags, country.value, "Event Label vs Visa Correlation")
            else:
                base = sent_best_view.filter(pl.col("country") == country.value)
                filtered = (
                    base.filter(pl.col("q_value") <= q_max.value)
                    .sort(["q_value", "neg_shock_lift", "pearson_corr"], descending=[False, True, True])
                    .head(top_n.value)
                )
                show_table(f"Sentiment signals | {country.value}", filtered, rows=top_n.value)
                show_country_heatmap(best_sentiment_lags, country.value, "Sentiment vs Visa Correlation")

            img_path = overlay_lookup.get(country.value)
            if img_path and Path(img_path).exists():
                print(f"\nOverlay plot: {img_path}")
                display(IPyImage(filename=img_path, width=1300))
            else:
                print("\nNo overlay image found for this country in overlay index.")

    mode.observe(_render, names="value")
    country.observe(_render, names="value")
    q_max.observe(_render, names="value")
    top_n.observe(_render, names="value")

    display(widgets.VBox([widgets.HBox([mode, country]), widgets.HBox([q_max, top_n]), output]))
    _render()

## Script Mode (Compute Layer)

This notebook is intentionally analysis-only.
Use the script to generate/refresh outputs:

`python src/analysis/event_visa_analysis.py --max-lag 6 --top-labels 8 --min-overlap 12 --min-event-months 12`

Output policy:
- Parquet for production/use datasets
- CSV for review/debug tables

This notebook then loads those outputs from `data/processed/test_outputs` and overlay images from `data/plots/events_vs_visas`.